## 1 — Project paths, inference settings, and imports

In [1]:
from pathlib import Path
import json
import re
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

# Run this notebook from the '45 CN' directory, or set PROJECT_DIR explicitly.
PROJECT_DIR = Path.cwd().resolve()
LATENCY_DIR = PROJECT_DIR / "latency_files"
CHECKPOINT_PATH = PROJECT_DIR / "ha2c_trained_models_65_nodes_lr_0_00001.pt"
TOPOLOGY_PATTERN = "65_CNs_topology_*.json"
OUTPUT_CSV = PROJECT_DIR / "ha2c_V8_65_latency_inference_results.csv"

EXPECTED_NODES = 65
MAX_STEPS = 200  # V8 trainer limit
LATENCY_LIMIT_MS = 10.0  # inclusive: <= 10 ms is feasible
LARGE_VALUE = 20.0       # same covered-column sentinel as training
FIXED_COSTS = np.array([5.0, 10.0, 20.0])
VARIABLE_COSTS_1 = np.array([[4.0, 8.0, 16.0], [1.0, 2.0, 4.0]])
VARIABLE_COSTS_2 = np.array([[2.0, 4.0, 8.0], [2.0, 4.0, 8.0]])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project folder:", PROJECT_DIR)
print("Latency JSON folder:", LATENCY_DIR)
print("Saved model:", CHECKPOINT_PATH)
print("Inference device:", DEVICE)


Project folder: /home/cci/Downloads/Latest/Revised/cost_and_inference_time_comparison/65 CN
Latency JSON folder: /home/cci/Downloads/Latest/Revised/cost_and_inference_time_comparison/65 CN/latency_files
Saved model: /home/cci/Downloads/Latest/Revised/cost_and_inference_time_comparison/65 CN/ha2c_trained_models_65_nodes_lr_0_00001.pt
Inference device: cuda


## 2 — Same 45-node actor architecture and V8 inference helpers

In [2]:
class Actor(nn.Module):
    """Actor layers and parameter names match the supplied V8 trainer.

    The trainer's FC size is 64*(state.shape[0]//8)**2; replace its
    notebook-global 'state' reference with the equivalent model node count.
    """
    def __init__(self, num_nodes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * (num_nodes // 8) * (num_nodes // 8), 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc4 = nn.Linear(512, num_nodes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x.unsqueeze(1))))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = x.reshape(x.shape[0], -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc4(x)  # raw logits (same logits as V8 Categorical)


def synchronize_if_cuda():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


def classify_nodes(cloud, regional, edge, node_count):
    """Same tier mapping as V8 encode_node_types; check exact coverage."""
    classes = {}
    for group, cls in ((cloud, 0), (regional, 1), (edge, 2)):
        for node in np.asarray(group).reshape(-1).astype(int).tolist():
            if node in classes:
                raise ValueError(f"Node {node} appears in multiple node tiers")
            classes[node] = cls
    if set(classes) != set(range(node_count)):
        raise ValueError(f"Node IDs must be 0..{node_count-1}; received {sorted(classes)}")
    return classes


def action_cost(node, active_nodes, component_costs, node_classes):
    """V8 compute_cost_new: fixed cost once, components on every action."""
    cls = node_classes[int(node)]
    fixed = 0.0 if int(node) in active_nodes else float(FIXED_COSTS[cls])
    variable = float(component_costs[:, cls].sum())
    return fixed + variable


@torch.inference_mode()
def highest_logit_valid_action(actor, working_state, valid_mask):
    """Use V8 valid-action logit mask; argmax is for deterministic evaluation."""
    if not bool(np.any(valid_mask)):
        raise RuntimeError("No valid placement action remains")
    state_tensor = torch.from_numpy(
        np.asarray(working_state, dtype=np.float32)
    ).unsqueeze(0).to(DEVICE)
    logits = actor(state_tensor).squeeze(0)
    if not bool(torch.isfinite(logits).all().item()):
        raise RuntimeError("Actor output contains NaN or infinity")
    valid = torch.as_tensor(valid_mask, dtype=torch.bool, device=DEVICE)
    masked_logits = logits.masked_fill(~valid, torch.finfo(logits.dtype).min)
    node_id = int(masked_logits.argmax().item())
    if not valid_mask[node_id]:
        raise RuntimeError("Action selection returned an invalid node")
    return node_id


def predict_agent(actor, initial_state, target_nodes, component_costs,
                  node_classes, previously_activated=()):
    """Sequential V8 action mask + column updates + fixed-cost sharing.

    Counts only the current agent's incremental placement cost. For Agent 2,
    previously_activated contains Agent 1's selected physical node IDs.
    """
    state_n = np.asarray(initial_state, dtype=np.float64).copy()
    targets = np.asarray(target_nodes, dtype=np.int64)
    if targets.size == 0:
        raise ValueError("Agent has no target nodes")
    remaining = set(int(t) for t in targets)
    active = set(int(n) for n in previously_activated)
    chosen = []
    episode_cost = 0.0

    for step in range(MAX_STEPS):
        if not remaining:
            break
        # V8: action must cover >=1 currently uncovered target, <=10 ms.
        valid_action_mask = np.any(
            state_n[:, targets] <= LATENCY_LIMIT_MS, axis=1
        )
        if not bool(valid_action_mask.any()):
            raise RuntimeError(f"Uncovered targets have no feasible action: {sorted(remaining)}")
        action = highest_logit_valid_action(actor, state_n, valid_action_mask)
        newly_covered = targets[state_n[action, targets] <= LATENCY_LIMIT_MS]
        new_targets = [int(t) for t in newly_covered if int(t) in remaining]
        if not new_targets:
            raise RuntimeError(f"Agent selected node {action} without additional target coverage")

        # V8 charges node activation once, including cross-agent reuse.
        episode_cost += action_cost(action, active, component_costs, node_classes)
        active.add(action)
        chosen.append(action)
        # Same update_state semantics: cover all newly satisfied target columns.
        state_n[:, new_targets] = LARGE_VALUE
        remaining.difference_update(new_targets)

    if remaining:
        raise RuntimeError(f"Reached {MAX_STEPS} decisions; uncovered targets: {sorted(remaining)}")
    return chosen, float(episode_cost)


## 3 — Find each latency JSON and load your saved 45-node actors

In [3]:
# Use the exact JSON preprocessor imported by your original trainer.
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"PROJECT_DIR does not exist: {PROJECT_DIR}")
if not LATENCY_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing latency_files folder: {LATENCY_DIR}. "
        "If your 45_CNs_topology_*.json files are actually in the project root, "
        "set LATENCY_DIR = PROJECT_DIR in cell 1 and rerun."
    )
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Saved model not found: {CHECKPOINT_PATH}")

sys.path.insert(0, str(PROJECT_DIR))
try:
    from data_extract_65_updated import process_topology
except ImportError as exc:
    raise ImportError(
        "Place the same data_extract_45_updated.py used by the trainer "
        f"inside {PROJECT_DIR}, or adjust sys.path accordingly."
    ) from exc


def latency_file_index(path):
    match = re.search(r"_(\d+)\.json$", path.name)
    if match is None:
        raise ValueError(f"Topology JSON filename does not end in _<index>.json: {path.name}")
    return int(match.group(1))


latency_files = sorted(LATENCY_DIR.glob(TOPOLOGY_PATTERN), key=latency_file_index)
if not latency_files:
    raise FileNotFoundError(f"No files matching {TOPOLOGY_PATTERN!r} in {LATENCY_DIR}")
indices = [latency_file_index(p) for p in latency_files]
if len(indices) != len(set(indices)):
    raise ValueError("More than one JSON has the same latency index")
if len(latency_files) != 30:
    print(f"NOTE: Found {len(latency_files)} latency JSON files; you expected 30.")

# Load only your own/trusted checkpoint; critic state is not used for inference.
try:
    saved = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except TypeError:  # PyTorch versions without the weights_only parameter
    saved = torch.load(CHECKPOINT_PATH, map_location="cpu")

actor1 = Actor(EXPECTED_NODES)
actor2 = Actor(EXPECTED_NODES)
for key, actor in (("actor1_state_dict", actor1), ("actor2_state_dict", actor2)):
    if key not in saved:
        raise KeyError(f"Missing {key!r} in {CHECKPOINT_PATH.name}")
    output_dim = int(saved[key]["fc4.weight"].shape[0])
    if output_dim != EXPECTED_NODES:
        raise ValueError(
            f"Checkpoint {key} has {output_dim} action outputs; "
            f"expected {EXPECTED_NODES}. Load the 45-node checkpoint, not the 50-node one."
        )
    actor.load_state_dict(saved[key], strict=True)
    actor.to(DEVICE).eval()

print(f"Loaded both 45-node actors; ready for {len(latency_files)} latency files.")
print("Indices:", indices)


Loaded both 45-node actors; ready for 30 latency files.
Indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


## 4 — Apply the V8 preprocessing and time each agent separately

In [4]:
def predict_one_latency_file(json_path):
    # 1. EXACT SAME PARSER as the V8 training notebook: DO NOT use raw json.load.
    cloud, regional, edge, raw_latency = process_topology(str(json_path))
    latency = np.asarray(raw_latency, dtype=np.float64)
    if latency.shape != (EXPECTED_NODES, EXPECTED_NODES):
        raise ValueError(
            f"Expected a {EXPECTED_NODES}x{EXPECTED_NODES} matrix in {json_path.name}; "
            f"got {latency.shape}."
        )
    if not np.isfinite(latency).all() or np.any(latency < 0):
        raise ValueError("Latency matrix must contain only nonnegative finite values")
    edge_nodes = np.asarray(edge, dtype=np.int64).reshape(-1)
    if edge_nodes.size == 0:
        raise ValueError("No E2/edge nodes returned by process_topology")
    if len(np.unique(edge_nodes)) != len(edge_nodes):
        raise ValueError("Duplicate E2/edge IDs returned by process_topology")
    node_classes = classify_nodes(cloud, regional, edge, EXPECTED_NODES)

    # 2. AGENT 1 INPUT — same as trainer's state1=state.copy() and
    # state1[:, all_non_edge_columns] = LARGE_VALUE.
    state1 = latency.copy()
    non_edge_cols = np.ones(EXPECTED_NODES, dtype=bool)
    non_edge_cols[edge_nodes] = False
    state1[:, non_edge_cols] = LARGE_VALUE

    # Time only Agent 1's sequential inference/placement loop.
    synchronize_if_cuda()
    start_agent1 = time.perf_counter()
    computing_nodes_list_1, cost1 = predict_agent(
        actor1, state1, edge_nodes, VARIABLE_COSTS_1, node_classes
    )
    synchronize_if_cuda()
    inference_time_agent1_s = time.perf_counter() - start_agent1

    selected_g1 = sorted(set(computing_nodes_list_1))
    # 3. EXACT SAME E2 -> G1 association as trainer: min original latency
    # among selected physical G1 nodes, not masked state1 values.
    e2_to_g1 = {}
    first_stage_latency = {}
    g1_to_e2 = {}
    for e2 in edge_nodes.tolist():
        g1 = min(selected_g1, key=lambda g: float(latency[g, e2]))
        hop1 = float(latency[g1, e2])
        if hop1 > LATENCY_LIMIT_MS:
            raise RuntimeError(f"E2 {e2}: no selected G1 with first-hop latency <=10 ms")
        e2_to_g1[int(e2)] = int(g1)
        first_stage_latency[int(e2)] = hop1
        g1_to_e2.setdefault(int(g1), []).append(int(e2))

    active_g1_nodes = sorted(g1_to_e2.keys())
    g1_first_stage_latency = {
        g1: max(first_stage_latency[e2] for e2 in attached_e2)
        for g1, attached_e2 in g1_to_e2.items()
    }

    # 4. AGENT 2 INPUT — same full-loop state as V8 trainer.
    state2 = np.full_like(latency, LARGE_VALUE, dtype=np.float64)
    for g1 in active_g1_nodes:
        state2[:, g1] = latency[:, g1] + g1_first_stage_latency[g1]
    for g1 in active_g1_nodes:
        if not bool(np.any(state2[:, g1] <= LATENCY_LIMIT_MS)):
            raise RuntimeError(f"G1 {g1}: no feasible G2 continuation <=10 ms")

    # Time only Agent 2's sequential inference/placement loop.
    # Agent 1's activated node IDs are carried into cost accounting.
    synchronize_if_cuda()
    start_agent2 = time.perf_counter()
    computing_nodes_list_2, cost2 = predict_agent(
        actor2, state2, active_g1_nodes, VARIABLE_COSTS_2,
        node_classes, previously_activated=computing_nodes_list_1
    )
    synchronize_if_cuda()
    inference_time_agent2_s = time.perf_counter() - start_agent2

    # 5. Verify each selected G1 has a feasible selected G2, then each
    # actual E2->G1->G2 path has total latency <= 10 ms.
    selected_g2 = sorted(set(computing_nodes_list_2))
    g1_to_g2 = {}
    for g1 in active_g1_nodes:
        feasible_g2 = [g2 for g2 in selected_g2
                       if float(latency[g2, g1]) + g1_first_stage_latency[g1]
                       <= LATENCY_LIMIT_MS]
        if not feasible_g2:
            raise RuntimeError(f"G1 {g1}: no selected G2 yields total latency <=10 ms")
        g1_to_g2[int(g1)] = int(min(feasible_g2,
            key=lambda g2: float(latency[g2, g1]) + g1_first_stage_latency[g1]))

    loop_latencies = [
        first_stage_latency[int(e2)]
        + float(latency[g1_to_g2[e2_to_g1[int(e2)]], e2_to_g1[int(e2)]])
        for e2 in edge_nodes.tolist()
    ]
    max_complete_loop_latency_ms = max(loop_latencies)
    if max_complete_loop_latency_ms > LATENCY_LIMIT_MS:
        raise RuntimeError(f"Final E2->G1->G2 latency {max_complete_loop_latency_ms} ms >10 ms")

    unique1 = sorted(set(computing_nodes_list_1))
    unique2 = sorted(set(computing_nodes_list_2))
    return {
        "status": "OK", "error": "",
        "cost1": float(cost1), "cost2": float(cost2),
        "cost_total": float(cost1 + cost2),
        "inference_time_agent1_s": float(inference_time_agent1_s),
        "inference_time_agent2_s": float(inference_time_agent2_s),
        "inference_time_total_s": float(inference_time_agent1_s + inference_time_agent2_s),
        "computing_nodes_list_1": [int(n) for n in computing_nodes_list_1],
        "computing_nodes_list_2": [int(n) for n in computing_nodes_list_2],
        "unique_nodes_list_1": unique1,
        "unique_nodes_list_2": unique2,
        "unique_nodes_list_combined": sorted(set(unique1) | set(unique2)),
        "shared_nodes_list": sorted(set(unique1) & set(unique2)),
        "num_unique_nodes_combined": len(set(unique1) | set(unique2)),
        "max_complete_loop_latency_ms": float(max_complete_loop_latency_ms),
        "e2_to_g1": e2_to_g1, "g1_to_g2": g1_to_g2,
    }


## 5 — Predict every latency file, display one row per index, and save the CSV

In [5]:
# Run every file independently; an unsuccessful file keeps its index and
# ERROR status in the output CSV, with NO fictitious total cost/time.
results = []
list_fields = [
    "computing_nodes_list_1", "computing_nodes_list_2",
    "unique_nodes_list_1", "unique_nodes_list_2",
    "unique_nodes_list_combined", "shared_nodes_list"
]
numeric_fields = [
    "cost1", "cost2", "cost_total", "inference_time_agent1_s",
    "inference_time_agent2_s", "inference_time_total_s",
    "num_unique_nodes_combined", "max_complete_loop_latency_ms"
]

for topology_file in latency_files:
    idx = latency_file_index(topology_file)
    try:
        result = predict_one_latency_file(topology_file)
        print(f"Latency {idx:02d}: cost1={result['cost1']:g}, cost2={result['cost2']:g}, "
              f"total={result['cost_total']:g}; "
              f"time1={result['inference_time_agent1_s']:.6f}s, "
              f"time2={result['inference_time_agent2_s']:.6f}s, "
              f"sum={result['inference_time_total_s']:.6f}s; "
              f"G1={result['computing_nodes_list_1']}, "
              f"G2={result['computing_nodes_list_2']}")
    except Exception as exc:
        result = {"status": "ERROR", "error": f"{type(exc).__name__}: {exc}"}
        result.update({field: None for field in numeric_fields})
        result.update({field: [] for field in list_fields})
        result.update({"e2_to_g1": {}, "g1_to_g2": {}})
        print(f"Latency {idx:02d}: ERROR: {result['error']}")
    results.append({"latency_index": idx, "topology_file": topology_file.name, **result})

results_df = pd.DataFrame(results)
# CSV-friendly JSON strings for list and mapping columns; reload with json.loads.
csv_df = results_df.drop(columns=["e2_to_g1", "g1_to_g2"]).copy()
for field in list_fields:
    csv_df[field] = csv_df[field].map(json.dumps)
csv_df.to_csv(OUTPUT_CSV, index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
display(results_df.drop(columns=["e2_to_g1", "g1_to_g2"]))
print(f"Saved CSV: {OUTPUT_CSV}")
print(f"Feasible results: {int((results_df['status'] == 'OK').sum())} / {len(results_df)}")


Latency 00: cost1=10, cost2=4, total=14; time1=1.046676s, time2=0.001803s, sum=1.048479s; G1=[0], G2=[0]
Latency 01: cost1=10, cost2=4, total=14; time1=0.001259s, time2=0.001058s, sum=0.002316s; G1=[0], G2=[0]
Latency 02: cost1=10, cost2=4, total=14; time1=0.001077s, time2=0.001000s, sum=0.002078s; G1=[0], G2=[0]
Latency 03: cost1=10, cost2=4, total=14; time1=0.001152s, time2=0.000993s, sum=0.002146s; G1=[0], G2=[0]
Latency 04: cost1=10, cost2=4, total=14; time1=0.001474s, time2=0.001377s, sum=0.002851s; G1=[0], G2=[0]
Latency 05: cost1=10, cost2=4, total=14; time1=0.001044s, time2=0.001054s, sum=0.002098s; G1=[0], G2=[0]
Latency 06: cost1=10, cost2=4, total=14; time1=0.001173s, time2=0.001016s, sum=0.002189s; G1=[0], G2=[0]
Latency 07: cost1=10, cost2=4, total=14; time1=0.001132s, time2=0.000995s, sum=0.002126s; G1=[0], G2=[0]
Latency 08: cost1=10, cost2=4, total=14; time1=0.001252s, time2=0.001737s, sum=0.002988s; G1=[0], G2=[0]
Latency 09: cost1=10, cost2=4, total=14; time1=0.001130

,latency_index,topology_file,status,error,cost1,cost2,cost_total,inference_time_agent1_s,inference_time_agent2_s,inference_time_total_s,computing_nodes_list_1,computing_nodes_list_2,unique_nodes_list_1,unique_nodes_list_2,unique_nodes_list_combined,shared_nodes_list,num_unique_nodes_combined,max_complete_loop_latency_ms
0,0,65_CNs_topology_0.json,OK,,10.0,4.0,14.0,1.046676,0.001803,1.048479,[0],[0],[0],[0],[0],[0],1,10.0
1,1,65_CNs_topology_1.json,OK,,10.0,4.0,14.0,0.001259,0.001058,0.002316,[0],[0],[0],[0],[0],[0],1,8.0
2,2,65_CNs_topology_2.json,OK,,10.0,4.0,14.0,0.001077,0.001000,0.002078,[0],[0],[0],[0],[0],[0],1,10.0
3,3,65_CNs_topology_3.json,OK,,10.0,4.0,14.0,0.001152,0.000993,0.002146,[0],[0],[0],[0],[0],[0],1,8.0
4,4,65_CNs_topology_4.json,OK,,10.0,4.0,14.0,0.001474,0.001377,0.002851,[0],[0],[0],[0],[0],[0],1,10.0
5,5,65_CNs_topology_5.json,OK,,10.0,4.0,14.0,0.001044,0.001054,0.002098,[0],[0],[0],[0],[0],[0],1,10.0
6,6,65_CNs_topology_6.json,OK,,10.0,4.0,14.0,0.001173,0.001016,0.002189,[0],[0],[0],[0],[0],[0],1,8.0
7,7,65_CNs_topology_7.json,OK,,10.0,4.0,14.0,0.001132,0.000995,0.002126,[0],[0],[0],[0],[0],[0],1,8.0
8,8,65_CNs_topology_8.json,OK,,10.0,4.0,14.0,0.001252,0.001737,0.002988,[0],[0],[0],[0],[0],[0],1,8.0
9,9,65_CNs_topology_9.json,OK,,10.0,4.0,14.0,0.001130,0.001012,0.002142,[0],[0],[0],[0],[0],[0],1,10.0


Saved CSV: /home/cci/Downloads/Latest/Revised/cost_and_inference_time_comparison/65 CN/ha2c_V8_65_latency_inference_results.csv
Feasible results: 30 / 30
